In [ ]:
import os
import torch
from datasets import load_dataset, Dataset
from PIL import Image
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import json

In [ ]:
current_dir = os.getcwd() 
dataset_dir = os.path.join(current_dir, "FinErva")

finerva_data_path = os.path.join(dataset_dir, "data-finerva-price", "data", "problems.json")
splits_path = os.path.join(dataset_dir, "data-finerva-price", "data", "pid_splits.json")
raw_image_folder = os.path.join(dataset_dir, "data-finerva-images", "images-price")

with open(finerva_data_path, 'r', encoding='utf-8') as f:
    raw_problems = json.load(f)
with open(splits_path, 'r', encoding='utf-8') as f:
    splits_data = json.load(f)

In [ ]:
train_ids = splits_data.get("train", [])
eval_ids = splits_data.get("val", []) 

In [ ]:
def extract_to_list(id_list):
    dataset_list = []
    missing_count = 0
    
    for pid in id_list:
        if pid not in raw_problems:
            continue
            
        item = raw_problems[pid]
        folder_id = pid 
        image_filename = item.get("image", "image.png")
        original_img_path = os.path.join(raw_image_folder, folder_id, image_filename)
        
        if not os.path.exists(original_img_path):
            missing_count += 1
            continue 
            
        choices = item.get("choices", [])
        answer_idx = item.get("answer", 0)
        choices_text = "\n".join([f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)])
        correct_answer = choices[answer_idx] if answer_idx < len(choices) else ""
        
        human_text = f"Ngữ cảnh: {item.get('lecture', '')}\n\nCâu hỏi: {item.get('question', '')}\nCác lựa chọn:\n{choices_text}\nHãy chọn đáp án đúng và giải thích."
        gpt_text = f"Đáp án đúng là: {correct_answer}.\nGiải thích: {item.get('solution', '')}"
        
        dataset_list.append({
            "id": pid,
            "image": original_img_path, 
            "conversations": [
                {"from": "human", "value": human_text},
                {"from": "gpt", "value": gpt_text}
            ]
        })
        
    if missing_count > 0:
        print(f"Bỏ qua {missing_count} mẫu do không tìm thấy file ảnh gốc.")
    return dataset_list

In [ ]:
# 2. Dùng hàm trích xuất (đã định nghĩa ở trên) để tạo Data theo chuẩn LLaVA
train_list = extract_to_list(train_ids)
test_list = extract_to_list(eval_ids)

# Chuyển thành định dạng Hugging Face Dataset
train_dataset = Dataset.from_list(train_list)
test_dataset = Dataset.from_list(test_list)

# 1. CẤU HÌNH LƯỢNG TỬ HÓA 4-BIT (Quantization)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
model_id = "llava-1.5-7b-hf"
processor = AutoProcessor.from_pretrained(model_id, local_files_only=True)

# 2. CHUẨN BỊ MÔ HÌNH CHO QLoRA

In [ ]:
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

In [ ]:
# Chuẩn bị mô hình cho việc train 4-bit trước khi bọc LoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
# Gắn Adapter LoRA vào mô hình gốc
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 3. TẢI DATASET

In [ ]:
class LlavaDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, instances):
        texts = []
        images = []
        
        for instance in instances:
            conv = instance['conversations']
            human_text = conv[0]['value']
            gpt_text = conv[1]['value']
            
            prompt = f"USER: <image>\n{human_text}\nASSISTANT: {gpt_text}"
            texts.append(prompt)
            
            img_path = instance['image']
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception as e:
                print(f"Cảnh báo: Lỗi đọc ảnh tại {img_path}. Đã thay bằng ảnh rỗng.")
                image = Image.new('RGB', (224, 224), color='black')
                
            images.append(image)
            
        batch = self.processor(
            text=texts, 
            images=images, 
            return_tensors="pt", 
            padding=True,
            truncation=True,
            max_length=1024 
        )
        
        batch["labels"] = batch["input_ids"].clone()
        batch["labels"][batch["attention_mask"] == 0] = -100 
        
        return batch

data_collator = LlavaDataCollator(processor)

# 4. THIẾT LẬP THAM SỐ HUẤN LUYỆN VÀ KHỞI CHẠY TRAINER

In [ ]:
output_dir = "./llava-stock-analyzer"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,             
    num_train_epochs=30,
    logging_steps=5,                
    

    save_strategy="epoch",
    per_device_eval_batch_size=2,

    optim="paged_adamw_8bit",
    fp16=False,                    
    bf16=True,                      
    max_grad_norm=1.0,              
    remove_unused_columns=False,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,                    
    train_dataset=train_dataset,    
    eval_dataset=eval_dataset,      
    data_collator=data_collator,
    args=training_args,             
)

In [ ]:
model.config.use_cache = False
trainer.train()

In [ ]:
trainer.save_model(f"{output_dir}/final")
processor.save_pretrained(f"{output_dir}/final")